In [ ]:
!git clone https://github.com/VladWero08/time-series-ad-gan.git

In [ ]:
pip install pandas numpy kagglehub torch scipy matplotlib

In [ ]:
import sys 
import os

module_path = os.path.abspath("./time-series-ad-gan")
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
import ast
import pandas as pd
import numpy as np
import json
import kagglehub
import typing as t
import torch
import matplotlib.pyplot as plt
from urllib.request import urlopen

from src.models.mad_gan import run_pipeline
from src.utils.data import intervals_to_points
from src.utils.errors import point_wise_error, area_wise_error, dtw_error

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Working on {device}!")

In [ ]:
def plot_signal(X: np.ndarray, y: np.ndarray) -> None:
    plt.figure(figsize=(15, 4))
    for idx in np.where(y == 1)[0]:
        plt.axvline(idx, color='red', alpha=0.4, linewidth=0.8, zorder=0)
    plt.plot(X, zorder=1)
    plt.show()

## **NAB**

In [ ]:
NAB_REPO = "https://raw.githubusercontent.com/numenta/NAB/refs/heads/master/data/"

ART_FOLDER = "artificialWithAnomaly/"
ART_FILE_NAMES = [
    "art_daily_flatmiddle",
    "art_daily_jumpsdown",
    "art_daily_jumpsup",
    "art_daily_nojump",
    "art_increase_spike_density",
    "art_load_balancer_spikes"
]

AD_EX_FOLDER = "realAdExchange/"
AD_EX_FILE_NAMES = [
    "exchange-2_cpc_results",
    "exchange-2_cpm_results",
    "exchange-3_cpc_results",
    "exchange-3_cpm_results",
    "exchange-4_cpc_results",
    "exchange-4_cpm_results",
]

AWS_FOLDER = "realAWSCloudwatch/"
AWS_FILE_NAMES = [
    "ec2_cpu_utilization_24ae8d",
    "ec2_cpu_utilization_53ea38",
    "ec2_cpu_utilization_5f5533",
    "ec2_cpu_utilization_77c1ca",
    "ec2_cpu_utilization_825cc2",
    "ec2_cpu_utilization_ac20cd",
    "ec2_cpu_utilization_c6585a",
    "ec2_cpu_utilization_fe7f93",
    "ec2_disk_write_bytes_1ef3de",
    "ec2_disk_write_bytes_c0d644",
    "ec2_network_in_257a54",
    "ec2_network_in_5abac7",
    "elb_request_count_8c0756",
    "grok_asg_anomaly",
    "iio_us-east-1_i-a2eb1cd9_NetworkIn",
    "rds_cpu_utilization_cc0c53",
    "rds_cpu_utilization_e47b3b"
]

TRAFFIC_FOLDER = "realTraffic/"
TRAFFIC_FILE_NAMES = [
    "TravelTime_387",
    "TravelTime_451",
    "occupancy_6005",
    "occupancy_t4013",
    "speed_6005",
    "speed_7578",
    "speed_t4013"
]

TWEETS_FOLDER = "realTweets/"
TWEETS_FILE_NAMES = [
    "Twitter_volume_AAPL",
    "Twitter_volume_AMZN",
    "Twitter_volume_CRM",
    "Twitter_volume_CVS",
    "Twitter_volume_FB",
    "Twitter_volume_GOOG",
    "Twitter_volume_IBM",
    "Twitter_volume_KO",
    "Twitter_volume_PFE",
    "Twitter_volume_UPS"
]

NAB_LABELS = "https://raw.githubusercontent.com/numenta/NAB/refs/heads/master/labels/combined_windows.json"

In [ ]:
NAB_HYPERPARAMS = {
    "train_ratio": 0.6,
    "device": device,
    "anomaly_type": "contextual",
    "verbose": True,
    "lr_g": 1e-5,
    "lr_d": 1e-5,
    "epochs": 1000,
    "rec_error_funcs": [("point", point_wise_error), ("area", area_wise_error)],
}

In [ ]:
with urlopen(NAB_LABELS) as labels_url:
    nab_labels = json.loads(labels_url.read().decode())

In [ ]:
def nab_download_subdataset(folder: str, file_names: list[str], nab_labels: dict) -> t.List[pd.DataFrame]:
    nab_signals = []
    nab_points = 0
    nab_anomalies = 0

    for i, fn in enumerate(file_names):
        # name of the .csv in the repository
        nab_ts_fn = f"{fn}.csv"
        # url to the raw .csv file in the repository
        nab_ts_url = f"{NAB_REPO}{folder}{nab_ts_fn}"

        nab_ts_df = pd.read_csv(nab_ts_url)
        nab_ts_df["is_anomaly"] = 0

        # iterate through the contextualized anomalies for each .csv file,
        # and if the timestamp is inside one of the intervals, toggle the is_anomaly attribute
        for start, end in nab_labels[f"{folder}{nab_ts_fn}"]:
            start = pd.Timestamp(start)
            end = pd.Timestamp(end)
            interval = pd.Interval(start, end, closed="both")

            nab_ts_df.loc[
                nab_ts_df["timestamp"].map(lambda ts: pd.Timestamp(ts) in interval),
                "is_anomaly"
            ] = 1

        nab_signals.append(nab_ts_df)
        nab_points += len(nab_ts_df)
        nab_anomalies += (nab_ts_df["is_anomaly"] == 1).sum()

        if (i + 1) % 5 == 0:
            print(f"Downloaded {i + 1} .csv files.")

    print()
    print(f"Finished downloading {folder}!")
    print("---------------------")
    print(f"Total Signals: {len(file_names)}")
    print(F"Total Points: {nab_points}")
    print(f"Total Anomaly Points: {nab_anomalies}")
    print(f"Anomaly Rate: {(nab_anomalies / nab_points) * 100:.2f}%")

    return nab_signals

### **Artificial with Anomaly**

In [ ]:
ART_HYPERPARAMS = NAB_HYPERPARAMS.copy()
ART_HYPERPARAMS["train_ratio"] = 0.5

In [ ]:
art_signals = nab_download_subdataset(folder=ART_FOLDER, file_names=ART_FILE_NAMES, nab_labels=nab_labels)

In [ ]:
total_metrics = {name: np.zeros(3) for (name, _) in ART_HYPERPARAMS["rec_error_funcs"]}

for i, art_signal in enumerate(art_signals):
    X = art_signal["value"].to_numpy()
    # expand from shape (T, ) to (T, 1)  
    X = np.expand_dims(X, axis=1)
    y = art_signal["is_anomaly"].to_numpy()
    
    print("--------------")
    print(f"Art Signal {i + 1}")
    print("--------------")

    metrics = run_pipeline(X, y, **ART_HYPERPARAMS)

    for key in total_metrics:
        total_metrics[key] += np.array(metrics[key])

print("------------------")
print("Art Signal Metrics")
print("------------------")

for key in total_metrics:
    total_metrics[key] = total_metrics[key] / len(art_signals)
    print(f"[{key}] Precision: {total_metrics[key][0]:.3f} | Recall: {total_metrics[key][1]:.3f} | F1: {total_metrics[key][2]:.3f}")

### **Ad Exchange**

In [ ]:
ad_ex_signals = nab_download_subdataset(folder=AD_EX_FOLDER, file_names=AD_EX_FILE_NAMES, nab_labels=nab_labels)

In [ ]:
total_metrics = {name: np.zeros(3) for (name, _) in NAB_HYPERPARAMS["rec_error_funcs"]}

for i, ad_ex_signal in enumerate(ad_ex_signals):
    X = ad_ex_signal['value'].to_numpy()
    # expand from shape (T, ) to (T, 1)  
    X = np.expand_dims(X, axis=1)
    # min-max normalization to [-1, 1]
    y = ad_ex_signal['is_anomaly'].to_numpy()
    
    print("--------------")
    print(f"Ad Exchange Signal {i + 1}")
    print("--------------")

    metrics = run_pipeline(X, y, **NAB_HYPERPARAMS)

    for key in total_metrics:
        total_metrics[key] += np.array(metrics[key])

print("------------------")
print("Ad Exchange Metrics")
print("------------------")

for key in total_metrics:
    total_metrics[key] = total_metrics[key] / len(ad_ex_signals)
    print(f"[{key}] Precision: {total_metrics[key][0]:.3f} | Recall: {total_metrics[key][1]:.3f} | F1: {total_metrics[key][2]:.3f}")

### **AWS Cloudwatch**

In [ ]:
aws_signals = nab_download_subdataset(folder=AWS_FOLDER, file_names=AWS_FILE_NAMES, nab_labels=nab_labels)

In [ ]:
total_metrics = {name: np.zeros(3) for (name, _) in NAB_HYPERPARAMS["rec_error_funcs"]}

for i, aws_signal in enumerate(aws_signals):
    X = aws_signal['value'].to_numpy()
    # expand from shape (T, ) to (T, 1)  
    X = np.expand_dims(X, axis=1)
    y = aws_signal['is_anomaly'].to_numpy()
    
    print("--------------")
    print(f"AWS Signal {i + 1}")
    print("--------------")

    metrics = run_pipeline(X, y, **NAB_HYPERPARAMS)

    for key in total_metrics:
        total_metrics[key] += np.array(metrics[key])

print("------------------")
print("AWS Metrics")
print("------------------")

for key in total_metrics:
    total_metrics[key] = total_metrics[key] / len(art_signals)
    print(f"[{key}] Precision: {total_metrics[key][0]:.3f} | Recall: {total_metrics[key][1]:.3f} | F1: {total_metrics[key][2]:.3f}")

### **Real Traffic**

In [ ]:
traffic_signals = nab_download_subdataset(folder=TRAFFIC_FOLDER, file_names=TRAFFIC_FILE_NAMES, nab_labels=nab_labels)

In [ ]:
total_metrics = {name: np.zeros(3) for (name, _) in NAB_HYPERPARAMS["rec_error_funcs"]}

for i, traffic_signal in enumerate(traffic_signals):
    X = traffic_signal['value'].to_numpy()
    # expand from shape (T, ) to (T, 1)  
    X = np.expand_dims(X, axis=1)
    y = traffic_signal['is_anomaly'].to_numpy()

    print("--------------")
    print(f"Traffic Signal {i + 1}")
    print("--------------")

    metrics = run_pipeline(X, y, **NAB_HYPERPARAMS)

    for key in total_metrics:
        total_metrics[key] += np.array(metrics[key])

print("------------------")
print("Real Traffic Metrics")
print("------------------")

for key in total_metrics:
    total_metrics[key] = total_metrics[key] / len(traffic_signals)
    print(f"[{key}] Precision: {total_metrics[key][0]:.3f} | Recall: {total_metrics[key][1]:.3f} | F1: {total_metrics[key][2]:.3f}")

### **Twitter**

In [ ]:
twitter_signals = nab_download_subdataset(folder=TWEETS_FOLDER, file_names=TWEETS_FILE_NAMES, nab_labels=nab_labels)

In [ ]:
total_metrics = {name: np.zeros(3) for (name, _) in NAB_HYPERPARAMS["rec_error_funcs"]}

for i, twitter_signal in enumerate(twitter_signals):
    X = twitter_signal['value'].to_numpy()
    # expand from shape (T, ) to (T, 1)  
    X = np.expand_dims(X, axis=1)
    y = twitter_signal['is_anomaly'].to_numpy()

    print("--------------")
    print(f"Twitter Signal {i + 1}")
    print("--------------")
    
    metrics = run_pipeline(X, y, **NAB_HYPERPARAMS)

    for key in total_metrics:
        total_metrics[key] += np.array(metrics[key])

print("------------------")
print("Twitter Metrics")
print("------------------")

for key in total_metrics:
    total_metrics[key] = total_metrics[key] / len(twitter_signals)
    print(f"[{key}] Precision: {total_metrics[key][0]:.3f} | Recall: {total_metrics[key][1]:.3f} | F1: {total_metrics[key][2]:.3f}")

## **Yahoo S5**

In [ ]:
A1_FOLDER = "https://raw.githubusercontent.com/harris0704/nbaData16-17/refs/heads/master/Yahoo_S5_Data/A1Benchmark/"
A1_FILE_NAME = "real_"
A1_N_FILES = 67

A2_FOLDER = "https://raw.githubusercontent.com/harris0704/nbaData16-17/refs/heads/master/Yahoo_S5_Data/A2Benchmark/"
A2_FILE_NAME = "synthetic_"
A2_N_FILES = 100

A3_FOLDER = "https://raw.githubusercontent.com/harris0704/nbaData16-17/refs/heads/master/Yahoo_S5_Data/A3Benchmark/"
A3_FILE_NAME = "A3Benchmark-TS"
A3_N_FILES = 100

A4_FOLDER = "https://raw.githubusercontent.com/harris0704/nbaData16-17/refs/heads/master/Yahoo_S5_Data/A4Benchmark/"
A4_FILE_NAME = "A4Benchmark-TS"
A4_N_FILES = 100

In [ ]:
YAHOO_HYPERPARAMS = {
    "train_ratio": 0.6,
    "device": device,
    "anomaly_type": "contextual",
    "verbose": False,
    "lr_g": 1e-5,
    "lr_d": 1e-5,
    "epochs": 1000,
    "rec_error_funcs": [("point", point_wise_error), ("area", area_wise_error)],
}

In [ ]:
def yahoo_download_subdataset(folder: str, file_name: str, n_files: int) -> t.List[pd.DataFrame]:
    yahoo_signals = []
    yahoo_points = 0
    yahoo_anomalies = 0

    for i in range(1, n_files + 1):
        # name of the .csv in the repository
        yahoo_fn = f"{file_name}{i}.csv"
        # url to the raw .csv file in the repository
        yahoo_url = f"{folder}{yahoo_fn}"
        yahoo_df = pd.read_csv(yahoo_url)
        yahoo_df = yahoo_df.rename(columns={"anomaly": "is_anomaly", "timestamps": "timestamp"})
        
        # count the number of points and the number of anomaly points
        yahoo_points += len(yahoo_df)
        yahoo_anomalies += (yahoo_df["is_anomaly"] == 1).sum()
        yahoo_signals.append(yahoo_df)

        if i % 10 == 0:
            print(f"Downloaded {i} .csv files.")

    print()
    print("Finished downloading!")
    print("---------------------")
    print(f"A1 Total Signals: {A1_N_FILES}")
    print(F"A1 Total Points: {yahoo_points}")
    print(f"A1 Total Anomaly Points: {yahoo_anomalies}")
    print(f"A1 Anomaly Rate: {(yahoo_anomalies / yahoo_points) * 100:.2f}%")

    return yahoo_signals

### **A1**

In [ ]:
a1_signals = yahoo_download_subdataset(folder=A1_FOLDER, file_name=A1_FILE_NAME, n_files=A1_N_FILES)

In [ ]:
total_metrics = {name: np.zeros(3) for (name, _) in YAHOO_HYPERPARAMS["rec_error_funcs"]}
total_evaluated = 0

for i, a1_signal in enumerate(a1_signals):
    X = a1_signal['value'].to_numpy()
    # expand from shape (T, ) to (T, 1)  
    X = np.expand_dims(X, axis=1)
    y = a1_signal['is_anomaly'].to_numpy()

    if (i + 1) % 5 == 0:
        print(f"A1 Signal {i + 1}...")

    try:
        metrics = run_pipeline(X, y, **YAHOO_HYPERPARAMS)
        for key in total_metrics:
            total_metrics[key] += np.array(metrics[key])
        total_evaluated += 1
    except:
        continue

print("------------------")
print("A1 Metrics")
print("------------------")

for key in total_metrics:
    total_metrics[key] = total_metrics[key] / total_evaluated
    print(f"[{key}] Precision: {total_metrics[key][0]:.3f} | Recall: {total_metrics[key][1]:.3f} | F1: {total_metrics[key][2]:.3f}")

### **A2**

In [ ]:
A2_HYPERPARAMS = YAHOO_HYPERPARAMS.copy()
A2_HYPERPARAMS["train_ratio"] = 0.5

In [ ]:
a2_signals = yahoo_download_subdataset(folder=A2_FOLDER, file_name=A2_FILE_NAME, n_files=A2_N_FILES)

In [ ]:
total_metrics = {name: np.zeros(3) for (name, _) in A2_HYPERPARAMS["rec_error_funcs"]}
total_evaluated = 0

for i, a2_signal in enumerate(a2_signals):
    X = a2_signal['value'].to_numpy()
    # expand from shape (T, ) to (T, 1)  
    X = np.expand_dims(X, axis=1)
    y = a2_signal['is_anomaly'].to_numpy()
    
    if (i + 1) % 5 == 0:
        print(f"A2 Signal {i + 1}...")
        
    try:
        metrics = run_pipeline(X, y, **A2_HYPERPARAMS)
        for key in total_metrics:
            total_metrics[key] += np.array(metrics[key])
        total_evaluated += 1
    except:
        continue

print("------------------")
print("A2 Metrics")
print("------------------")

for key in total_metrics:
    total_metrics[key] = total_metrics[key] / total_evaluated
    print(f"[{key}] Precision: {total_metrics[key][0]:.3f} | Recall: {total_metrics[key][1]:.3f} | F1: {total_metrics[key][2]:.3f}")

### **A3**

In [ ]:
A3_HYPERPARAMS = YAHOO_HYPERPARAMS.copy()
A3_HYPERPARAMS["train_ratio"] = 0.5

In [ ]:
a3_signals = yahoo_download_subdataset(folder=A3_FOLDER, file_name=A3_FILE_NAME, n_files=A3_N_FILES)

In [ ]:
total_metrics = {name: np.zeros(3) for (name, _) in A3_HYPERPARAMS["rec_error_funcs"]}
total_evaluated = 0

for i, a3_signal in enumerate(a3_signals):
    X = a3_signal['value'].to_numpy()
    # expand from shape (T, ) to (T, 1)  
    X = np.expand_dims(X, axis=1)
    y = a3_signal['is_anomaly'].to_numpy()
    
    if (i + 1) % 5 == 0:
        print(f"A3 Signal {i + 1}...")

    try:
        metrics = run_pipeline(X, y, **A3_HYPERPARAMS)
        for key in total_metrics:
            total_metrics[key] += np.array(metrics[key])
        total_evaluated += 1
    except:
        continue

print("------------------")
print("A3 Metrics")
print("------------------")

for key in total_metrics:
    total_metrics[key] = total_metrics[key] / total_evaluated
    print(f"[{key}] Precision: {total_metrics[key][0]:.3f} | Recall: {total_metrics[key][1]:.3f} | F1: {total_metrics[key][2]:.3f}")

### **A4**

In [ ]:
A4_HYPERPARAMS = YAHOO_HYPERPARAMS.copy()
A4_HYPERPARAMS["train_ratio"] = 0.5

In [ ]:
a4_signals = yahoo_download_subdataset(folder=A4_FOLDER, file_name=A4_FILE_NAME, n_files=A4_N_FILES)

In [ ]:
total_metrics = {name: np.zeros(3) for (name, _) in A4_HYPERPARAMS["rec_error_funcs"]}
total_evaluated = 0

for i, a4_signal in enumerate(a4_signals):
    X = a4_signal['value'].to_numpy()
    # expand from shape (T, ) to (T, 1)  
    X = np.expand_dims(X, axis=1)
    y = a4_signal['is_anomaly'].to_numpy()
    
    if (i + 1) % 5 == 0:
        print(f"A4 Signal {i + 1}...")

    try:
        metrics = run_pipeline(X, y, **A4_HYPERPARAMS)
        for key in total_metrics:
            total_metrics[key] += np.array(metrics[key])
        total_evaluated += 1
    except:
        continue

print("------------------")
print("A4 Metrics")
print("------------------")

for key in total_metrics:
    total_metrics[key] = total_metrics[key] / total_evaluated
    print(f"[{key}] Precision: {total_metrics[key][0]:.3f} | Recall: {total_metrics[key][1]:.3f} | F1: {total_metrics[key][2]:.3f}")

## **NASA**

In [ ]:
NASA_SPLIT_REPO = "https://raw.githubusercontent.com/ML4ITS/mtad-gat-pytorch/refs/heads/main/datasets/data/"
NASA_LABELS_FILE = "labeled_anomalies.csv"
NASA_MSL_SPLIT = "msl_train_md.csv"
NASA_SMAP_SPLIT = "smap_train_md.csv"

In [ ]:
NASA_HYPERPARAMS = {
    "device": device,
    "anomaly_type": "contextual",
    "verbose": False,
    "lr_g": 1e-5,
    "lr_d": 1e-5,
    "epochs": 1000,
    "rec_error_funcs": [("point", point_wise_error), ("area", area_wise_error)],
}

In [ ]:
nasa_dataset_path = kagglehub.dataset_download("patrickfleith/nasa-anomaly-detection-dataset-smap-msl")
# path to the training samples from the Kaggle dataset
nasa_train_path = os.path.join(nasa_dataset_path, "data", "data", "train")
# path to the testing samples from the Kaggle dataset
nasa_test_path = os.path.join(nasa_dataset_path, "data", "data", "test")

In [ ]:
# download the NASA labels that correspond to the labels for test data
nasa_labels_url = f"{NASA_SPLIT_REPO}{NASA_LABELS_FILE}"
nasa_labels = pd.read_csv(nasa_labels_url)

### **MSL**

In [ ]:
MSL_HYPERPARAMS = NASA_HYPERPARAMS.copy()
MSL_HYPERPARAMS["verbose"] = True

In [ ]:
msl_split_url = f"{NASA_SPLIT_REPO}{NASA_MSL_SPLIT}"
msl_split = pd.read_csv(msl_split_url)

# extract the channel names corresponding to the MSL dataset
msl_channel_names = list(msl_split["chan_id"].values) 
# add the .npy file extensions to the MSL file names
msl_file_names = [f"{msl_file_name}.npy" for msl_file_name in msl_channel_names]
# sort them alphabetically
msl_file_names.sort()

In [ ]:
msl_train: list[np.ndarray] = []
msl_test: list[np.ndarray] = []
msl_test_y: list[list] = []

for msl_file_name in msl_file_names:
    # load both the train and test time series for the same channel, concatenate
    # the time series and add them to the MSL dataset
    msl_train_ts = np.load(os.path.join(nasa_train_path, msl_file_name))
    msl_test_ts = np.load(os.path.join(nasa_test_path, msl_file_name))

    # extract the anomaly labels for the test time-series
    # -4 is needed to get rid of the file extension from the name
    row = nasa_labels[nasa_labels["chan_id"] == msl_file_name[:-4]]
    msl_test_interval_str = row["anomaly_sequences"].iloc[0]
    msl_test_interval = ast.literal_eval(msl_test_interval_str)
    msl_test_labels = intervals_to_points(msl_test_interval, len(msl_test_ts))

    msl_train.append(msl_train_ts)
    msl_test.append(msl_test_ts)

    msl_test_y.append(msl_test_labels)

In [ ]:
total_metrics = {name: np.zeros(3) for (name, _) in MSL_HYPERPARAMS["rec_error_funcs"]}
total_evaluated = 0

for msl_channel, X_train, X_test, y_test in zip(msl_channel_names, msl_train, msl_test, msl_test_y):
    # all data points are considered to be normal in the training tensor,
    # thus the labels consist of a tensor full of zeros
    y_train = torch.zeros(X_train.shape[0])

    print(f"Channel {msl_channel} Signal...")

    X_train = np.expand_dims(X_train[:, 0], axis=1)
    X_test = np.expand_dims(X_test[:, 0], axis=1)
    
    try:
        metrics = run_pipeline(
            X=X_train, y=y_train,
            X_test=X_test, y_test=y_test,
            **MSL_HYPERPARAMS
        )
        for key in total_metrics:
            total_metrics[key] += np.array(metrics[key])
        total_evaluated += 1
    except:
        continue

print("------------------")
print("MSL Metrics")
print("------------------")

for key in total_metrics:
    total_metrics[key] = total_metrics[key] / total_evaluated
    print(f"[{key}] Precision: {total_metrics[key][0]:.3f} | Recall: {total_metrics[key][1]:.3f} | F1: {total_metrics[key][2]:.3f}")

### **SMAP**

In [ ]:
SMAP_HYPERPARAMS = NASA_HYPERPARAMS.copy()
SMAP_HYPERPARAMS["verbose"] = False

In [ ]:
smap_split_url = f"{NASA_SPLIT_REPO}{NASA_SMAP_SPLIT}"
smap_split = pd.read_csv(smap_split_url)

# extract the file names corresponding to the SMAP dataset
smap_channels_names = list(smap_split["chan_id"].values)
# add the .npy file extensions to the SMAP file names
smap_file_names = [f"{smap_file_name}.npy" for smap_file_name in smap_channels_names]
# sort them alphabetically
smap_file_names.sort()

In [ ]:
smap_train: list[np.ndarray] = []
smap_test: list[np.ndarray] = []
smap_test_y: list[list] = []

for smap_file_name in smap_file_names:
    # load both the train and test time series for the same channel, concatenate
    # the time series and add them to the MSL dataset
    smap_train_ts = np.load(os.path.join(nasa_train_path, smap_file_name))
    smap_test_ts = np.load(os.path.join(nasa_test_path, smap_file_name))

    # extract the anomaly labels for the test time-series
    # -4 is needed to get rid of the file extension from the name
    row = nasa_labels[nasa_labels["chan_id"] == smap_file_name[:-4]]
    smap_test_interval_str = row["anomaly_sequences"].iloc[0]
    smap_test_interval = ast.literal_eval(smap_test_interval_str)
    smap_test_labels = intervals_to_points(smap_test_interval, len(smap_test_ts))

    smap_train.append(smap_train_ts)
    smap_test.append(smap_test_ts)

    smap_test_y.append(smap_test_labels)

In [ ]:
total_metrics = {name: np.zeros(3) for (name, _) in SMAP_HYPERPARAMS["rec_error_funcs"]}
total_evaluated = 0

for smap_channel, X_train, X_test, y_test in zip(smap_channels_names, smap_train, smap_test, smap_test_y):
    # all data points are considered to be normal in the training tensor,
    # thus the labels consist of a tensor full of zeros
    y_train = torch.zeros(X_train.shape[0])

    print(f"Channel {smap_channel} Signal...")

    X_train = np.expand_dims(X_train[:, 0], axis=1)
    X_test = np.expand_dims(X_test[:, 0], axis=1)
    
    try:
        metrics = run_pipeline(
            X=X_train, y=y_train,
            X_test=X_test, y_test=y_test,
            **SMAP_HYPERPARAMS
        )
        for key in total_metrics:
            total_metrics[key] += np.array(metrics[key])
        total_evaluated += 1
    except:
        continue

print("------------------")
print("SMAP Metrics")
print("------------------")

for key in total_metrics:
    total_metrics[key] = total_metrics[key] / total_evaluated
    print(f"[{key}] Precision: {total_metrics[key][0]:.3f} | Recall: {total_metrics[key][1]:.3f} | F1: {total_metrics[key][2]:.3f}")